# Trend-Based Strategy Experiment
This notebook focuses on evaluating the Trend-Based (Rule-Based) strategy using momentum indicators.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Ensure plots look nice
plt.style.use('ggplot')

In [ ]:
# Load Multi-Asset Data
file_path = 'real_crypto_data.csv'
try:
    df = pd.read_csv(file_path, parse_dates=['Date'], index_col='Date')
    print("Loaded Assets:", df.columns.tolist())
except FileNotFoundError:
    print("File not found. Generating dummy.")
    dates = pd.date_range(start='2023-01-01', periods=1095)
    df = pd.DataFrame(np.random.randn(1095, 5), index=dates, columns=['BTC','ETH','XRP','LTC','BCH'])
    df = (1 + df.pct_change().fillna(0)).cumprod() * 100

def get_market_features(df):
    returns = df.pct_change().fillna(0)
    volatility = returns.rolling(window=20).std().fillna(0)
    sma50 = df.rolling(window=50).mean()
    momentum = (df / sma50 - 1).fillna(0)
    return returns, volatility, momentum

returns_df, vol_df, mom_df = get_market_features(df)
assets = df.columns.tolist()
n_assets = len(assets)

# Define Training Range (2023 and 2024)
train_mask = (df.index.year == 2023) | (df.index.year == 2024)
train_df_slice = df[train_mask]
train_returns_slice = returns_df[train_mask]
train_vol_slice = vol_df[train_mask]
train_mom_slice = mom_df[train_mask]

# Define Test Range (2025 Only)
test_mask = (df.index.year == 2025)
test_df_slice = df[test_mask]

print(f"Total Data Points: {len(df)}")
print(f"Training Data (2023-2024): {len(train_df_slice)} days")
print(f"Testing Data (2025): {len(test_df_slice)} days")

In [ ]:
# Helper for Markowitz Optimization
def optimize_markowitz(returns_window, cov_matrix):
    n = returns_window.shape[1]
    if n == 0: return []
    def neg_sharpe(weights):
        fed_return = np.sum(returns_window.mean() * weights) * 252
        fed_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
        return -(fed_return / fed_vol) if fed_vol > 0 else 0
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(n))
    init_guess = n * [1. / n]
    try:
        res = minimize(neg_sharpe, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
        return res.x
    except:
        return init_guess

In [ ]:
def run_trend_based_evaluation(eval_df, eval_returns, eval_vol, eval_mom):
    print(f"\n{'='*10} Starting Trend-Based Strategy Evaluation (Year 2025) {'='*10}")
    if len(eval_df) < 10:
        return None

    # Portfolio tracking
    portfolio_values = {
        'Trend-Based (Rule-Based)': [100.0],
        'Buy and Hold (Market Baseline)': [100.0]
    }
    bnh_holdings = np.ones(n_assets) * (100.0 / n_assets) 
    lookback = 50

    for step in range(len(eval_df) - 1):
        current_date = eval_df.index[step]
        
        # Prepare Market Data
        if step + 1 < len(eval_returns):
            day_returns = eval_returns.iloc[step + 1].values
        else:
            day_returns = np.zeros(n_assets)

        # Calculate Markowitz weights for base allocation
        glob_idx = returns_df.index.get_loc(current_date)
        if glob_idx >= lookback:
            window_ret = returns_df.iloc[glob_idx-lookback:glob_idx]
            cov_mat = window_ret.cov().values
            mw_weights = optimize_markowitz(window_ret, cov_mat)
        else:
            mw_weights = np.ones(n_assets) / n_assets
        
        # Trend-Based Strategy
        current_mom = np.mean(eval_mom.iloc[step].values)
        if current_mom > 0: 
            trend_weights = mw_weights  # Bullish: full allocation
        else: 
            trend_weights = 0.5 * mw_weights  # Bearish: defensive (50% allocation)
        ret_trend = np.sum(trend_weights * day_returns)
        portfolio_values['Trend-Based (Rule-Based)'].append(
            portfolio_values['Trend-Based (Rule-Based)'][-1] * (1 + ret_trend)
        )
        
        # Buy and Hold (Market Baseline)
        bnh_holdings = bnh_holdings * (1 + day_returns)
        portfolio_values['Buy and Hold (Market Baseline)'].append(np.sum(bnh_holdings))
        
    return portfolio_values

In [ ]:
# Run Evaluation
eval_returns = returns_df[test_mask]
eval_vol = vol_df[test_mask]
eval_mom = mom_df[test_mask]

portfolio_values = run_trend_based_evaluation(test_df_slice, eval_returns, eval_vol, eval_mom)

In [ ]:
# Calculate Performance Metrics
def calculate_metrics(portfolio_values):
    metrics = {}
    for name, values in portfolio_values.items():
        values_array = np.array(values)
        total_return = (values_array[-1] / values_array[0] - 1) * 100
        
        # Calculate daily returns
        daily_returns = np.diff(values_array) / values_array[:-1]
        volatility = np.std(daily_returns) * np.sqrt(252) * 100
        
        # Sharpe Ratio (assuming risk-free rate = 0)
        mean_return = np.mean(daily_returns) * 252
        sharpe = mean_return / (np.std(daily_returns) * np.sqrt(252)) if np.std(daily_returns) > 0 else 0
        
        metrics[name] = {
            'Total Return': f"{total_return:.2f}%",
            'Volatility': f"{volatility:.2f}%",
            'Sharpe Ratio': f"{sharpe:.2f}"
        }
    
    return pd.DataFrame(metrics).T

results_df = calculate_metrics(portfolio_values)
print("\n>>> TREND-BASED STRATEGY RESULTS (2025) <<<")
print(results_df)

In [ ]:
# Plot Portfolio Performance
plt.figure(figsize=(14, 7))
for name, values in portfolio_values.items():
    plt.plot(values, label=name, linewidth=2)

plt.title('Trend-Based Strategy vs Buy and Hold (2025)', fontsize=16, fontweight='bold')
plt.xlabel('Trading Days', fontsize=12)
plt.ylabel('Portfolio Value ($)', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed Analysis: Momentum Signal Distribution
print("\n=== Momentum Signal Analysis ===")
eval_mom_mean = eval_mom.mean(axis=1)
bullish_days = (eval_mom_mean > 0).sum()
bearish_days = (eval_mom_mean <= 0).sum()

print(f"Total Trading Days: {len(eval_mom_mean)}")
print(f"Bullish Days (momentum > 0): {bullish_days} ({bullish_days/len(eval_mom_mean)*100:.1f}%)")
print(f"Bearish Days (momentum <= 0): {bearish_days} ({bearish_days/len(eval_mom_mean)*100:.1f}%)")

# Plot momentum distribution
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(eval_mom_mean, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Line')
plt.title('Distribution of Average Momentum', fontsize=14, fontweight='bold')
plt.xlabel('Momentum Value', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(eval_mom_mean.values, linewidth=1.5)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Zero Line')
plt.title('Momentum Over Time (2025)', fontsize=14, fontweight='bold')
plt.xlabel('Trading Days', fontsize=12)
plt.ylabel('Average Momentum', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()